# Classificação de gêneros com Word2Vec

In [1]:
from IPython.display import display

Carregamento, preparação do texto e criação dos rótulos

In [2]:
import re
import unicodedata
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

CAMINHO_CSV = "gutenberg_books.csv"
SEMENTE = 42
TEST_SIZE = 0.25
MIN_AMOSTRAS_POR_GENERO = 1


def normalizar_para_regra(texto):
    texto = "" if pd.isna(texto) else str(texto).lower()
    texto = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in texto if not unicodedata.combining(c))


def identificar_genero(subjects):
    s = normalizar_para_regra(subjects)

    regras = [
        # Categorias originais
        (
            "Ficção científica/fantasia",
            [
                r"science fiction",
                r"fantasy fiction",
                r"fantasy stories",
                r"utopias?",
                r"dystopias?",
            ],
        ),
        (
            "Mistério/policial",
            [
                r"detective and mystery stories",
                r"detective stories",
                r"mystery fiction",
                r"crime fiction",
                r"police stories",
                r"suspense fiction",
            ],
        ),
        (
            "Terror/gótico",
            [
                r"horror fiction",
                r"horror tales?",
                r"ghost stories",
                r"gothic fiction",
                r"supernatural fiction",
                r"vampire fiction",
                r"paranormal fiction",
            ],
        ),
        (
            "Romance",
            [
                r"love stories",
                r"romance fiction",
                r"romantic fiction",
                r"courtship.{0,30}fiction",
            ],
        ),
        (
            "Guerra",
            [
                r"war stories",
                r"war fiction",
                r"military fiction",
                r"world war.{0,30}fiction",
            ],
        ),
        (
            "Aventura",
            [
                r"adventure stories",
                r"adventure fiction",
                r"sea stories",
                r"pirates.{0,30}fiction",
            ],
        ),
        (
            "Ficção histórica",
            [
                r"historical fiction",
                r"historical stories",
            ],
        ),
        (
            "Faroeste",
            [
                r"western stories",
                r"cowboy stories",
                r"western fiction",
            ],
        ),

        # Novas categorias identificadas no CSV
        (
            "Romance de formação",
            [
                r"\bbildungsromans?\b",
            ],
        ),
        (
            "Ficção doméstica/familiar",
            [
                r"\bdomestic fiction\b",
            ],
        ),
        (
            "Ficção psicológica",
            [
                r"\bpsychological fiction\b",
            ],
        ),
        (
            "Humor",
            [
                r"\bhumorous stories?\b",
            ],
        ),
        (
            "Ficção didática",
            [
                r"\bdidactic fiction\b",
            ],
        ),
        (
            "Ficção autobiográfica",
            [
                r"\bautobiographical fiction\b",
            ],
        ),
        (
            "Ficção epistolar",
            [
                r"\bepistolary fiction\b",
            ],
        ),
        (
            "Ficção política",
            [
                r"\bpolitical fiction\b",
            ],
        ),
        (
            "Romance literário",
            [
                r"\bromances\b",
            ],
        ),
        (
            "Literatura infantil/juvenil",
            [
                r"\bjuvenile fiction\b",
                r"\bchildren['’]s stories\b",
            ],
        ),
        (
            "Poesia",
            [
                r"\bpoetry\b",
                r"\bpoems?\b",
                r"\bsonnets?\b",
            ],
        ),
    ]

    for genero, padroes in regras:
        if any(re.search(padrao, s) for padrao in padroes):
            return genero

    return np.nan


df_original = pd.read_csv(CAMINHO_CSV, dtype={"book_id": str})
print(f"Livros carregados: {len(df_original)}")
print("Colunas disponíveis:", list(df_original.columns))

if "text_final" in df_original.columns:
    coluna_texto = "text_final"
elif "summary" in df_original.columns:
    coluna_texto = "summary"
elif "trecho_1_clean" in df_original.columns:
    coluna_texto = "trecho_1_clean"
else:
    raise ValueError(
        "Não encontrei 'text_final', 'summary' nem 'trecho_1_clean' no CSV. "
        "Confira o arquivo gutenberg_books.csv."
    )

df = df_original.copy()
df["texto"] = df[coluna_texto].fillna("").astype(str).str.strip()

if "genero" in df.columns and df["genero"].notna().any():
    df["genero"] = df["genero"].astype(str).str.strip()
    df.loc[df["genero"].str.lower().isin(["", "nan", "none", "null"]), "genero"] = np.nan
else:
    if "subjects" not in df.columns:
        raise ValueError(
            "O CSV não tem gêneros anotados na coluna 'genero' nem metadados 'subjects'. "
            "Adicione uma coluna 'genero' manualmente ou recupere 'subjects' no scraping."
        )
    df["genero"] = df["subjects"].apply(identificar_genero)

df = df[(df["texto"] != "") & df["genero"].notna()].copy()

contagem = df["genero"].value_counts()
generos_validos = contagem[contagem >= MIN_AMOSTRAS_POR_GENERO].index
df = df[df["genero"].isin(generos_validos)].reset_index(drop=True)

print(f"Coluna textual usada: {coluna_texto}")
print(f"Livros com texto e rótulo identificável: {len(df)}")
print("Distribuição dos gêneros aproximados:")
print(df["genero"].value_counts())

if df["genero"].nunique() < 2:
    raise ValueError(
        "Foram encontrados menos de dois gêneros com pelo menos "
        f"{MIN_AMOSTRAS_POR_GENERO} livros cada. Verifique a distribuição de 'subjects' "
        "ou crie/adicione uma coluna 'genero' manualmente. As regras são heurísticas e "
        "podem precisar ser adaptadas aos valores reais de subjects."
    )

X_texto = df["texto"].to_numpy()
y = df["genero"].to_numpy()
indices = np.arange(len(df))
idx_treino, idx_teste = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=SEMENTE,
    stratify=y,
)

texto_treino = df.iloc[idx_treino]["texto"].tolist()
texto_teste = df.iloc[idx_teste]["texto"].tolist()
y_treino = df.iloc[idx_treino]["genero"].to_numpy()
y_teste = df.iloc[idx_teste]["genero"].to_numpy()
print(f"Exemplos para treino: {len(texto_treino)} | para teste: {len(texto_teste)}")


Livros carregados: 312
Colunas disponíveis: ['book_id', 'title', 'author', 'language', 'loc_class', 'subjects', 'release_date', 'ebook_url', 'summary', 'work_key', 'trecho', 'summary_normalized', 'summary_clean', 'tokens', 'tokens_no_punctuation', 'tokens_final', 'text_final', 'tokens_lemmatized', 'text_lemmatized', 'tokens_stemmed', 'text_stemmed']
Coluna textual usada: text_final
Livros com texto e rótulo identificável: 218
Distribuição dos gêneros aproximados:
genero
Mistério/policial              48
Romance                        35
Ficção científica/fantasia     31
Aventura                       17
Terror/gótico                  14
Romance de formação            11
Romance literário               8
Guerra                          8
Literatura infantil/juvenil     7
Poesia                          6
Ficção doméstica/familiar       5
Ficção histórica                5
Ficção psicológica              5
Humor                           5
Ficção didática                 4
Ficção epistola

Aplicação da representação

In [3]:
from gensim.models import Word2Vec

tokens_treino = [texto.lower().split() for texto in texto_treino]
tokens_teste = [texto.lower().split() for texto in texto_teste]

modelo_w2v = Word2Vec(
    sentences=tokens_treino,
    vector_size=100,
    window=5,
    min_count=2,
    workers=1,
    sg=1,
    epochs=20,
)

def embedding_livro(tokens, modelo):
    vetores = [modelo.wv[token] for token in tokens if token in modelo.wv]
    if not vetores:
        return np.zeros(modelo.vector_size)
    return np.mean(vetores, axis=0)

X_treino = np.array([embedding_livro(tokens, modelo_w2v) for tokens in tokens_treino])
X_teste = np.array([embedding_livro(tokens, modelo_w2v) for tokens in tokens_teste])

print("Dimensão dos embeddings de treino:", X_treino.shape)
print("Dimensão dos embeddings de teste:", X_teste.shape)
print("Tamanho do vocabulário Word2Vec:", len(modelo_w2v.wv))


Dimensão dos embeddings de treino: (163, 100)
Dimensão dos embeddings de teste: (55, 100)
Tamanho do vocabulário Word2Vec: 2087


Treinamento e avaliação

In [4]:
classificador = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEMENTE)
classificador.fit(X_treino, y_treino)
predicoes = classificador.predict(X_teste)

print(f"Acurácia: {accuracy_score(y_teste, predicoes):.3f}")
print(f"F1 macro: {f1_score(y_teste, predicoes, average='macro', zero_division=0):.3f}")
print("\nRelatório de classificação:")
print(classification_report(y_teste, predicoes, zero_division=0))

rotulos = sorted(set(y_teste) | set(predicoes))
matriz_confusao = confusion_matrix(y_teste, predicoes, labels=rotulos)
display(pd.DataFrame(matriz_confusao, index=rotulos, columns=rotulos).rename_axis("Real \\ Previsto"))


Acurácia: 0.036
F1 macro: 0.012

Relatório de classificação:
                             precision    recall  f1-score   support

                   Aventura       0.00      0.00      0.00         4
                   Faroeste       0.00      0.00      0.00         1
 Ficção científica/fantasia       0.00      0.00      0.00         8
            Ficção didática       0.00      0.00      0.00         1
  Ficção doméstica/familiar       0.00      0.00      0.00         1
           Ficção epistolar       0.00      0.00      0.00         1
           Ficção histórica       0.00      0.00      0.00         1
            Ficção política       0.00      0.00      0.00         1
         Ficção psicológica       0.00      0.00      0.00         1
                     Guerra       0.00      0.00      0.00         2
                      Humor       0.00      0.00      0.00         1
Literatura infantil/juvenil       0.07      0.50      0.12         2
          Mistério/policial       0.00   

,Aventura,Faroeste,Ficção científica/fantasia,Ficção didática,Ficção doméstica/familiar,Ficção epistolar,Ficção histórica,Ficção política,Ficção psicológica,Guerra,Humor,Literatura infantil/juvenil,Mistério/policial,Poesia,Romance,Romance de formação,Romance literário,Terror/gótico
Real \ Previsto,,,,,,,,,,,,,,,,,,
Aventura,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,1,1
Faroeste,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
Ficção científica/fantasia,0,1,0,0,0,0,0,0,0,0,0,2,0,0,0,1,0,4
Ficção didática,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
Ficção doméstica/familiar,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
Ficção epistolar,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
Ficção histórica,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
Ficção política,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
Ficção psicológica,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
